# **FEATURE EXTRACTION**

In [1]:
import os
import torch
import numpy as np
import time
import sys
import re
!pip install facenet_pytorch

py_path = "/content/drive/MyDrive/11/Consegna finale/src"

if py_path not in sys.path:
    sys.path.append(py_path)

from config import *
from feature_extractor import FaceRecognitionSystem
from feature_extractor import GalleryDataset
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

## **Test Inferenza Real-Time e Validazione**

Questa sezione esegue la simulazione operativa del sistema di riconoscimento facciale:

1.  **Building Gallery:** Le immagini di riferimento vengono convertite in embeddings tramite *InceptionResnetV1* e caricate in **VRAM** per un accesso immediato.
2.  **Inferenza Singola:** Per ogni immagine di test (Probe), il sistema:
    * Estrae il feature vector.
    * Calcola la **Distanza Euclidea** ($L_2$) rispetto a tutti i vettori della Gallery.
    * Identifica il *Nearest Neighbor* (il volto con distanza minore).
3.  **Metriche:** Vengono monitorati in tempo reale l'accuratezza, la latenza (in secondi) e l'allocazione di memoria GPU.

In [2]:
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
system = FaceRecognitionSystem(DEVICE)

system.build_and_save_gallery(GALLERY_DIR, FEATURES_SAVE_DIR)

print("\n--- Inizio Test Inferenza Singola (Simulazione Real-Time) ---")

probe_files = []
for root, _, files in os.walk(PROBES_DIR):
    for file in files:
        if file.lower().endswith(('.jpg', '.png', '.jpeg')):
            true_label_match = re.search(r'\d+', os.path.basename(root))
            true_label = int(true_label_match.group()) if true_label_match else -1

            probe_files.append((os.path.join(root, file), true_label))

probe_files.sort()

correct_count = 0
total_count = 0

for img_path, true_label in probe_files:

    start_time = time.time()

    pred_label, distance = system.predict_single(img_path)

    elapsed = time.time() - start_time

    is_correct = (pred_label == true_label)
    if is_correct: correct_count += 1
    total_count += 1

    print(f"Probe: {os.path.basename(img_path)} | Pred: {pred_label} (Vero: {true_label}) | Dist: {distance:.4f} | Time: {elapsed:.3f}s")

    if total_count % 10 == 0:
        system.print_memory()

if total_count > 0:
    acc = (correct_count / total_count) * 100
    print(f"\n--- FINE TEST ---")
    print(f"Accuratezza Totale: {acc:.2f}% ({correct_count}/{total_count})")

Inizializzazione Modello su cuda:0...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.


  0%|          | 0.00/107M [00:00<?, ?B/s]

TypeError: FaceRecognitionSystem.build_and_save_gallery() missing 1 required positional argument: 'prefix_labels'